# 08. Introduction to Numba (JIT Compilation)

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand what Numba is and how JIT compilation works
- Accelerate Python code using Numba decorators
- Compare performance between regular Python and Numba-accelerated code
- Know when to use Numba for optimization

## 🔗 Where this fits

**Builds on:** Course 05 — Unit 1, lesson 02 "pandas & NumPy Basics" — vectorisation is the first speed-up; Numba is what you reach for when the loop cannot be vectorised.

**Used later in:** Course 05 — Unit 5, lesson 06, which profiles code before deciding what to accelerate.

---


**All concepts are explained in the code comments below - you can learn everything from this notebook alone!**

---

## 🔗 Solving the Performance Problem

**Remember the performance challenge?**
- Python loops can be slow for numerical computations
- NumPy helps, but sometimes we need more speed
- We need a way to accelerate Python code without rewriting in C/C++

**This notebook solves that problem!**
- We'll learn **Numba** - Just-In-Time (JIT) compilation for Python
- We'll see Python code run 10-100x faster with simple decorators
- We'll understand when JIT compilation helps vs when it doesn't

**This enables high-performance Python code without leaving Python!**

---

## The Story: Turbo Mode

Imagine your Python code is a car. Normal Python runs at normal speed. **Numba** is like adding a turbo - same car, same code, but 10-100x faster! You just add a decorator (like turning on turbo mode) and your code runs much faster.

---

## Why Numba Matters

Numba is crucial for:
- **Numerical Computations**: Accelerate loops and math operations
- **Performance-Critical Code**: When speed matters more than flexibility
- **Scientific Computing**: Fast numerical Python without C/C++
- **Data Science**: Speed up data processing pipelines

## Learning Objectives
1. Understand what Numba is and JIT compilation
2. Use @numba.jit decorator to accelerate functions
3. Compare performance: regular Python vs Numba
4. Know when Numba helps and when it doesn't
5. Apply Numba to real numerical computations


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Numba JIT, pure Python vs compiled
- Libraries: numba, numpy

**Outputs:** What you'll see when you run the cells

- Timing comparisons

---

In [1]:
# Step 1: Import necessary libraries
import numpy as np
import time

# Try to import Numba (optional - works without GPU)
try:
    from numba import jit
    NUMBA_AVAILABLE = True
    print("✅ Numba imported successfully!")
except ImportError:
    NUMBA_AVAILABLE = False
    print("⚠️  Numba not available. Install with: pip install numba")
    print("   This notebook will show the concept but won't run Numba code.")
    # Create a dummy decorator for demonstration
    def jit(*args, **kwargs):
        def decorator(func):
            return func
        return decorator

✅ Numba imported successfully!


In [2]:
# Step 2: Regular Python function (slow) - for large arrays
def slow_sum(arr):
    total = 0.0
    for i in range(len(arr)):
        total += arr[i]
    return total

# Step 3: Numba-accelerated function (fast)
@jit(nopython=True)
def fast_sum(arr):
    total = 0.0
    for i in range(len(arr)):
        total += arr[i]
    return total

# Fallback: if Numba not available, use slow_sum for both
if not NUMBA_AVAILABLE:
    fast_sum = slow_sum

print("✅ Functions defined!")

✅ Functions defined!


Functions defined above. **Next:** performance comparison (regular Python vs Numba).


In [3]:
# WHAT: Time the same summation loop in pure Python and in Numba, on a real column of network-traffic data.
# WHY: JIT compilation only proves itself on a real workload - here, totalling every byte sent in a 2.3-million-flow capture.

# Load ONE real numeric column out of the 707 MB CIC-IDS2017 capture.
# Why one column? Because that is all this benchmark needs - and usecols makes
# reading a huge file affordable.
import pandas as pd
DATA_DIR = '../../../Course 04/datasets/raw/'

print("📊 Loading a real numeric column (bytes sent per network flow)...")
fwd_bytes = pd.read_csv(DATA_DIR + 'cicids2017.csv',
                        usecols=['Total Length of Fwd Packets'])

# .astype('float64') so both functions do identical floating-point work.
large_array = fwd_bytes['Total Length of Fwd Packets'].values.astype('float64')
print(f"   Loaded {large_array.size:,} real values")
print(f"   These are forward-direction byte counts for every captured flow.")

if NUMBA_AVAILABLE:
    # Time regular Python
    start = time.time()
    result_slow = slow_sum(large_array)
    time_slow = time.time() - start

    # Time Numba (first call compiles, so it's slower)
    start = time.time()
    result_fast = fast_sum(large_array)
    time_fast_first = time.time() - start

    # Time Numba (second call - already compiled, very fast!)
    start = time.time()
    result_fast = fast_sum(large_array)
    time_fast = time.time() - start

    print(f"\nTotal bytes sent across all flows: {result_slow:,.0f}")
    print(f"\nRegular Python: {time_slow:.4f} seconds")
    print(f"Numba (first call - compilation): {time_fast_first:.4f} seconds")
    print(f"Numba (second call - compiled): {time_fast:.4f} seconds")
    print(f"\nSpeedup after compilation: {time_slow / time_fast:.1f}x faster!")
    print(f"\nResults match: {np.isclose(result_slow, result_fast)}")
    print("\n💡 Note the first call: compilation is NOT free. Numba pays for itself")
    print("   only when the compiled function is called many times, or on data")
    print("   large enough that the compile cost disappears next to the work.")
else:
    print("Numba not available - install with: pip install numba")
    print("With Numba, you would see a large speedup on this numerical loop.")


📊 Loading a real numeric column (bytes sent per network flow)...


   Loaded 2,300,825 real values
   These are forward-direction byte counts for every captured flow.



Total bytes sent across all flows: 1,272,795,262

Regular Python: 0.1325 seconds
Numba (first call - compilation): 0.2063 seconds
Numba (second call - compiled): 0.0015 seconds

Speedup after compilation: 89.1x faster!

Results match: True

💡 Note the first call: compilation is NOT free. Numba pays for itself
   only when the compiled function is called many times, or on data
   large enough that the compile cost disappears next to the work.


In [4]:
# WHAT: Print practical guidance on when Numba helps and when it does not.
# WHY: JIT compilation shines on numerical loops but cannot speed up string- or object-heavy code - knowing the limits saves wasted effort.

# Step 5: When to use Numba
print("=" * 70)
print("When to Use Numba:")
print("=" * 70)
print("✅ Good for:")
print("   - Numerical loops (for, while)")
print("   - NumPy array operations")
print("   - Mathematical computations")
print("   - Performance-critical code")
print("\n❌ Not good for:")
print("   - String operations")
print("   - Pandas DataFrames (use cuDF instead)")
print("   - Code with Python objects")
print("   - Code that changes frequently")
print("\n💡 Tip: Use Numba for numerical bottlenecks, not everything!")

When to Use Numba:
✅ Good for:
   - Numerical loops (for, while)
   - NumPy array operations
   - Mathematical computations
   - Performance-critical code

❌ Not good for:
   - String operations
   - Pandas DataFrames (use cuDF instead)
   - Code with Python objects
   - Code that changes frequently

💡 Tip: Use Numba for numerical bottlenecks, not everything!


## 📚 References

1. Lam, S. K., Pitrou, A., & Seibert, S. (2015). *Numba: A LLVM-based Python JIT Compiler*. Proceedings of the Second Workshop on the LLVM Compiler Infrastructure in HPC. <https://doi.org/10.1145/2833157.2833162>
2. Harris, C. R., Millman, K. J., van der Walt, S. J., et al. (2020). *Array Programming with NumPy*. Nature, 585, 357-362. <https://arxiv.org/abs/2006.10256>
3. van der Walt, S., Colbert, S. C., & Varoquaux, G. (2011). *The NumPy Array: A Structure for Efficient Numerical Computation*. Computing in Science & Engineering, 13(2), 22-30. <https://arxiv.org/abs/1102.1523>